In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))


In [3]:

from src.data.ingest import build_universe, download_prices, clean_prices

tickers = build_universe()
cache = Path("../data/raw/prices.parquet")
raw_df = download_prices(tickers, cache)
clean_df, report = clean_prices(raw_df)


Loading cached data from ../data/raw/prices.parquet
Dropping 7 tickers: ['AAF.L', 'CCEP.L', 'EDV.L', 'HLN.L', 'MNG.L', 'MTLN.L', 'PSH.L']


In [4]:
from src.features.engineer import build_features

features_df = build_features(clean_df)
print(features_df.shape)
print(features_df.head())
print(features_df.isnull().sum())


(271467, 13)
                   momentum  return_1m  volatility_21d  drawdown_52w  \
date       ticker                                                      
2015-01-02 AAL.L        NaN        NaN             NaN           NaN   
           ABF.L        NaN        NaN             NaN           NaN   
           ADM.L        NaN        NaN             NaN           NaN   
           ALW.L        NaN        NaN             NaN           NaN   
           ANTO.L       NaN        NaN             NaN           NaN   

                   ma_ratio_200  rsi_14  volume_ratio_20  beta_252  \
date       ticker                                                    
2015-01-02 AAL.L            NaN     NaN              NaN       NaN   
           ABF.L            NaN     NaN              NaN       NaN   
           ADM.L            NaN     NaN              NaN       NaN   
           ALW.L            NaN     NaN              NaN       NaN   
           ANTO.L           NaN     NaN              NaN      

In [5]:
features_clean = features_df.dropna()
print(features_clean.shape)
print(features_clean.index.get_level_values("date").min())


(245862, 13)
2015-12-24 00:00:00


In [6]:
features_clean.to_parquet("../data/processed/features.parquet")
print("Saved.")


Saved.


In [7]:
corr_check = features_clean[["ma_ratio_200", "rsi_14"]].groupby(level="ticker").corr()
# Extract just the cross-correlation (not self-correlation)
cross_corr = corr_check.unstack()["ma_ratio_200"]["rsi_14"]
median_abs_corr = cross_corr.abs().median()
print(f"Median |correlation| between ma_ratio_200 and rsi_14: {median_abs_corr:.3f}")


Median |correlation| between ma_ratio_200 and rsi_14: 0.588


In [8]:
features_clean.columns

Index(['momentum', 'return_1m', 'volatility_21d', 'drawdown_52w',
       'ma_ratio_200', 'rsi_14', 'volume_ratio_20', 'beta_252',
       'relative_strength', 'percentile_52w', 'vix', 'forward_return',
       'forward_volatility'],
      dtype='object')

In [9]:
features_clean = features_df.dropna()
print(features_clean.shape)
print(features_clean.columns.tolist())
features_clean.to_parquet("../data/processed/features.parquet")


(245862, 13)
['momentum', 'return_1m', 'volatility_21d', 'drawdown_52w', 'ma_ratio_200', 'rsi_14', 'volume_ratio_20', 'beta_252', 'relative_strength', 'percentile_52w', 'vix', 'forward_return', 'forward_volatility']


In [11]:
from scipy.stats import spearmanr
import itertools
import pandas as pd

feature_cols = [
    "momentum", "return_1m", "volatility_21d", "drawdown_52w",
    "ma_ratio_200", "rsi_14", "volume_ratio_20", "beta_252", "vix",
    "relative_strength", "percentile_52w"
]

# Compute per-ticker Spearman correlation matrices
ticker_corrs = []
for ticker, group in features_clean[feature_cols].groupby(level="ticker"):
    corr = group.droplevel("ticker").corr(method="spearman")
    ticker_corrs.append(corr.abs())

# Median |corr| across tickers
median_corr = pd.concat(ticker_corrs).groupby(level=0).median()

print("Full 11×11 median |Spearman| correlation matrix:")
print(median_corr.round(3).to_string())

print("\n--- Specific pairs ---")
pairs = [
    ("ma_ratio_200",     "rsi_14"),
    ("relative_strength","momentum"),
    ("percentile_52w",   "drawdown_52w"),
    ("percentile_52w",   "ma_ratio_200"),
]
for a, b in pairs:
    print(f"{a:25s} vs {b:25s}: {median_corr.loc[a,b]:.3f}")

print("\n--- Pairs above 0.7 ---")
for a, b in itertools.combinations(feature_cols, 2):
    val = median_corr.loc[a, b]
    if val > 0.7:
        print(f"  {a} vs {b}: {val:.3f}")


Full 11×11 median |Spearman| correlation matrix:
                   momentum  return_1m  volatility_21d  drawdown_52w  ma_ratio_200  rsi_14  volume_ratio_20  beta_252    vix  relative_strength  percentile_52w
beta_252              0.224      0.050           0.226         0.193         0.148   0.072            0.028     1.000  0.220              0.043           0.160
drawdown_52w          0.645      0.453           0.402         1.000         0.874   0.566            0.020     0.193  0.310              0.403           0.947
ma_ratio_200          0.572      0.491           0.292         0.874         1.000   0.581            0.025     0.148  0.223              0.444           0.921
momentum              1.000      0.073           0.271         0.645         0.572   0.069            0.027     0.224  0.196              0.054           0.642
percentile_52w        0.642      0.504           0.320         0.947         0.921   0.605            0.026     0.160  0.254              0.452        